# Load the packages

In [1]:
import numpy as np
import pandas as pd
import cv2
import datetime
import glob

from detect import detect_objects, logData, frame_bb, save

# Load the data

In [2]:
seq = 3

currentPath = "C:/Users/shaia/Documents/Opgaveregning/AutoSys/4. semester/Perception for autonome systemer/Eksamprojekt/"
dataPathLeft = currentPath + f"34759_final_project_rect/seq_0{seq}/image_02/"
dataPathRight = currentPath + f"34759_final_project_rect/seq_0{seq}/image_03/"
dataPNGLeft = dataPathLeft + "data/"
dataPNGRight = dataPathRight + "data/"
dataTime = dataPathRight + "timestamps.txt"

dataLeft = glob.glob(f"{dataPNGLeft}*.png")
dataRight = glob.glob(f"{dataPNGRight}*.png")

timeStamp = open(dataTime)
timeData = timeStamp.readlines()
timeStamp.close()
time1 = datetime.datetime.strptime(timeData[0][:-4], "%Y-%m-%d %H:%M:%S.%f")
time2 = datetime.datetime.strptime(timeData[1][:-4], "%Y-%m-%d %H:%M:%S.%f")
deltaT = (time2 - time1).total_seconds()

# Make the Kalman filter

In [3]:
states = 4

x0 = np.zeros((3 * states, 1))
u = np.zeros(x0.shape)
P0car = np.diag([100000, 100, 0.1] * states)
P0ped = np.diag([100000, 0.01, 0.000001] * states)
P0cyc = np.diag([100000, 0.01, 0.0001] * states)
R = 0.0001 * np.eye(states)

F = np.eye(x0.shape[0])
for i in range(F.shape[0]):
    try:
        F[i, i + 1] = deltaT if i % 3 != 2 and i < F.shape[0] - 1 else 0
        F[i, i + 2] = 0.5 * deltaT ** 2 if i % 3 == 0 and i < F.shape[0] - 2 else 0
    except:
        pass

H = np.zeros((R.shape[0], x0.shape[0]))
for i in range(H.shape[0]):
    H[i, i * 3] = 1
    
kalman0 = {"x": x0, "u": u, "Pcar": P0car, "Pped": P0ped, "Pcyc": P0cyc, "F": F, "H": H, "R": R}

# Find the bounding box and get the dataframe

In [4]:
bboxLeft = detect_objects(dataLeft)
dfLeft = logData(dataLeft, bboxLeft, kalman0)

bboxRight = detect_objects(dataRight)
dfRight = logData(dataRight, bboxRight, kalman0)

videoLeft = []
for i in range(max(dfLeft["frame"])):
    img = cv2.imread(dataLeft[i])
    
    if i not in dfLeft["frame"]:
        videoLeft.append(img)
        continue
    
    subData = dfLeft.loc[dfLeft["frame"] == i]
    coords = np.array([subData["bbox left"], subData["bbox top"], subData["bbox right"], subData["bbox bottom"]]).T
    types = list(subData["type"])
    trackID = list(subData["track id"])

    imgFrame = frame_bb(img, coords, types, trackID)
    videoLeft.append(imgFrame)

videoRight = []
for i in range(max(dfRight["frame"])):
    img = cv2.imread(dataRight[i])
    
    if i not in dfRight["frame"]:
        videoRight.append(img)
        continue
    
    subData = dfRight.loc[dfRight["frame"] == i]
    coords = np.array([subData["bbox left"], subData["bbox top"], subData["bbox right"], subData["bbox bottom"]]).T
    types = list(subData["type"])
    trackID = list(subData["track id"])

    imgFrame = frame_bb(img, coords, types, trackID)
    videoRight.append(imgFrame)


0: 224x640 3 cars, 2 pedestrians, 1 cyclist, 100.0ms
Speed: 2.8ms preprocess, 100.0ms inference, 2.9ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 3 cars, 2 pedestrians, 44.3ms
Speed: 2.0ms preprocess, 44.3ms inference, 2.1ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 3 cars, 2 pedestrians, 45.3ms
Speed: 1.7ms preprocess, 45.3ms inference, 2.5ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 3 cars, 2 pedestrians, 42.8ms
Speed: 1.5ms preprocess, 42.8ms inference, 2.0ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 3 cars, 1 pedestrian, 40.9ms
Speed: 1.6ms preprocess, 40.9ms inference, 1.5ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 2 cars, 1 pedestrian, 1 cyclist, 44.9ms
Speed: 1.5ms preprocess, 44.9ms inference, 1.5ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 2 cars, 1 pedestrian, 1 cyclist, 43.7ms
Speed: 1.8ms preprocess, 43.7ms inference, 1.8ms postprocess per image at shape (1, 3, 224, 

# Do something here

# Save the image and CSV file

In [5]:
save(videoLeft, dfLeft, 1 / deltaT, f"ResultSeq{seq}Left")
save(videoRight, dfRight, 1 / deltaT, f"ResultSeq{seq}Right")